# Spike Model Fitting

Encode training data with `multidms.Data` and fit models across a grid of
fusion regularization values for each replicate.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import warnings
import os
import time
import pickle

import pandas as pd
import multidms

from notebooks._common import load_config, build_fit_params, robust_fit_models

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]
seed = cfg["seed"]
fit = spike["fitting"]

output_dir = spike["output_dir"]
reference = spike["reference"]
condition_colors = spike["condition_colors"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})

# Optionally subsample
subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=seed)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
    print(f"Subsampled to {len(func_score_df)} rows")

print(f"Loaded {len(func_score_df)} training variants")

## Create Data objects

In [ ]:
cc = condition_colors
datasets = []
for res, fsdf in func_score_df.groupby("replicate"):
    start = time.time()
    df_agg = (
        fsdf.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")
    data = multidms.Data(
        df_agg,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=reference,
        assert_site_integrity=False,
        verbose=True,
        name=f"rep-{res}",
    )
    data.condition_colors = cc
    datasets.append(data)
    print(f"Replicate {res}: {round(time.time() - start)}s")

## Configure and fit models

In [ ]:
fit_params = build_fit_params(fit, datasets)
print("Fitting parameters:")
for k, v in fit_params.items():
    if k != "dataset":
        print(f"  {k}: {v}")

In [ ]:
_, _, models = robust_fit_models(
    fit_params, gpu_ids=gpu_ids, n_processes=n_processes
)

# Convert dict-valued columns to strings for groupby compatibility
for col in models.columns:
    if models[col].apply(lambda x: isinstance(x, dict)).any():
        models[col] = models[col].apply(str)

models["replicate"] = models.dataset_name.str.split("-").str[-1].astype(int)
print(f"Fit {len(models)} models")

In [ ]:
pickle.dump(models, open(f"{output_dir}/full_models.pkl", "wb"))
print(f"Saved {output_dir}/full_models.pkl")